# Segmentation Evaluation

Evaluate MicroAtlas segmentation performance on 18 public test datasets using Average Precision (AP) at IoU thresholds 0.5, 0.75, and 0.9.

**Metrics:**
- AP@0.5, AP@0.75, AP@0.9 — Average Precision at different IoU overlap thresholds
- mAP — mean AP across the three thresholds

**Test data:** `data/all/test/` containing 18 dataset subdirectories with `.tif` images and `_mask.tif` ground truth annotations.

In [ ]:
import sys
from pathlib import Path

# Ensure src/ is on the import path
SRC_DIR = Path.cwd().parent / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from segment_anything import sam_model_registry
from cellpose import io, metrics, models, utils, train, transforms, dynamics
import time
from tqdm import trange
from torch import nn
import torch.nn.functional as F
import torch
import numpy as np
from natsort import natsorted
import os
import pandas as pd
from PIL import Image

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device('cuda')
io.logger_setup()

## Load MicroAtlas Model

In [ ]:
model = models.CellposeModel(gpu=True, pretrained_model='./microatlas/microatlas')
net = model.net
net.eval()
print(f'Model loaded on {device}')

## Evaluation Function

In [ ]:
def eval_subsets(root, dataset_name, save_results=True, save_dir_base='./data/all/predicted_masks_per_dataset'):
    """
    Evaluate model performance on a single dataset.
    Returns: (ap_mean, per_image_aps, image_names)
    """
    test_files = root.glob('*.tif')
    test_files = natsorted([tf for tf in test_files if '_mask' not in str(tf)])
    print(f'  nimg_test = {len(test_files)}')

    if len(test_files) == 0:
        print(f'  No test files found in {root}')
        return None, None, None

    # Load images
    test_data = []
    for i in trange(len(test_files), desc='Loading images'):
        img = io.imread(test_files[i])
        if len(img.shape) == 2:
            img = np.tile(img[np.newaxis, :, :], (3, 1, 1))
            img[1:] = 0
        test_data.append(img)

    # Load ground truth masks
    test_masks = [io.imread(str(test_files[i])[:-4] + '_mask.tif') for i in trange(len(test_files), desc='Loading masks')]

    # Run segmentation
    bsize = 256
    diameter = 30.
    masks_pred = model.eval(test_data, diameter=diameter, channels=None, niter=1000,
                            batch_size=256, bsize=bsize)[0]

    # Save predicted masks as PNG
    if save_results:
        save_dir = os.path.join(save_dir_base, f'{dataset_name}_microatlas')
        os.makedirs(save_dir, exist_ok=True)
        for i in range(len(masks_pred)):
            base_name = Path(test_files[i]).stem
            save_path = os.path.join(save_dir, f'{base_name}_pred.png')
            mask_img = masks_pred[i].astype(np.uint8)
            if mask_img.max() > 0:
                mask_img = (mask_img / mask_img.max() * 255).astype(np.uint8)
            Image.fromarray(mask_img).save(save_path)

    # Compute AP per image
    masks_gt = [tl.astype('uint16') for tl in test_masks]
    threshold = np.arange(0.5, 1.0, 0.05)
    image_names = []
    per_image_aps = []

    for i in range(len(test_files)):
        base_name = Path(test_files[i]).stem
        image_names.append(base_name)
        ap_i, tp_i, fp_i, fn_i = metrics.average_precision(
            [masks_gt[i]], [masks_pred[i]], threshold=threshold
        )
        epsilon = 1e-10
        for n in range(len(tp_i)):
            denominator = tp_i[n] + fp_i[n] + fn_i[n] + epsilon
            ap_i[n] = tp_i[n] / denominator
        per_image_aps.append(ap_i[0])

    per_image_aps = np.array(per_image_aps)

    # Compute overall AP
    ap, tp, fp, fn = metrics.average_precision(masks_gt, masks_pred, threshold=threshold)
    epsilon = 1e-10
    for n in range(len(tp)):
        denominator = tp[n] + fp[n] + fn[n] + epsilon
        ap[n] = tp[n] / denominator

    ap_mean = ap[:, [0, 5, 8]].mean(axis=0)
    print(f'  {dataset_name}: AP@0.5={ap_mean[0]:.4f}  AP@0.75={ap_mean[1]:.4f}  AP@0.9={ap_mean[2]:.4f}')

    return ap_mean, per_image_aps, image_names

## Run Evaluation on All Test Datasets

In [ ]:
test_root = Path('./data/all/test/')
test_root_ls = sorted(os.listdir(test_root))

all_dataset_names = []
all_image_names_dict = {}
all_per_image_aps_dict = {}

for dataset_name in test_root_ls:
    root = test_root / dataset_name
    print(f'\nEvaluating: {dataset_name}')
    ap_mean, per_image_aps, image_names = eval_subsets(
        root, dataset_name=dataset_name, save_results=True
    )
    if ap_mean is not None:
        all_dataset_names.append(dataset_name)
        all_image_names_dict[dataset_name] = image_names
        all_per_image_aps_dict[dataset_name] = per_image_aps

## Save Results to Excel

In [ ]:
if len(all_dataset_names) > 0:
    excel_path = './data/all/microatlas.xlsx'

    # Find maximum number of images across datasets
    max_images = max(len(all_per_image_aps_dict[dn]) for dn in all_dataset_names)

    threshold_indices = {0.5: 0, 0.75: 5, 0.9: 8}

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        for thresh_name, thresh_idx in threshold_indices.items():
            data = {'Dataset': all_dataset_names}
            for img_idx in range(max_images):
                col_name = str(img_idx + 1)
                img_aps = []
                for dn in all_dataset_names:
                    aps = all_per_image_aps_dict[dn]
                    if img_idx < len(aps):
                        img_aps.append(aps[img_idx, thresh_idx])
                    else:
                        img_aps.append(np.nan)
                data[col_name] = img_aps

            df = pd.DataFrame(data)
            sheet_name = f'map{thresh_name}'
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f'Saved {sheet_name} sheet')

    print(f'\nExcel file saved to: {excel_path}')

## Summary

In [ ]:
# Print summary table
print(f"{'Dataset':<25s} {'AP@0.5':>8s} {'AP@0.75':>8s} {'AP@0.9':>8s} {'mAP':>8s}")
print('-' * 60)

# Re-run to collect per-dataset AP means for summary
summary_records = []
for dataset_name in test_root_ls:
    root = test_root / dataset_name
    test_files = root.glob('*.tif')
    test_files = natsorted([tf for tf in test_files if '_mask' not in str(tf)])
    if len(test_files) == 0:
        continue

    test_data = []
    for tf in test_files:
        img = io.imread(tf)
        if len(img.shape) == 2:
            img = np.tile(img[np.newaxis, :, :], (3, 1, 1))
            img[1:] = 0
        test_data.append(img)
    test_masks = [io.imread(str(tf)[:-4] + '_mask.tif') for tf in test_files]
    masks_gt = [tl.astype('uint16') for tl in test_masks]

    masks_pred = model.eval(test_data, diameter=30., channels=None, niter=1000,
                            batch_size=256, bsize=256)[0]

    threshold = np.arange(0.5, 1.0, 0.05)
    ap, tp, fp, fn = metrics.average_precision(masks_gt, masks_pred, threshold=threshold)
    epsilon = 1e-10
    for n in range(len(tp)):
        ap[n] = tp[n] / (tp[n] + fp[n] + fn[n] + epsilon)
    ap_mean = ap[:, [0, 5, 8]].mean(axis=0)
    map_score = ap_mean.mean()
    summary_records.append({
        'Dataset': dataset_name,
        'AP@0.5': ap_mean[0],
        'AP@0.75': ap_mean[1],
        'AP@0.9': ap_mean[2],
        'mAP': map_score,
    })
    print(f'{dataset_name:<25s} {ap_mean[0]:>8.4f} {ap_mean[1]:>8.4f} {ap_mean[2]:>8.4f} {map_score:>8.4f}')

df_summary = pd.DataFrame(summary_records)
print(f'\nOverall mAP: {df_summary["mAP"].mean():.4f}')